# clip-grad-norm-pre-step — ex1: clip grads to a max global L2 norm before optimizer.step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `clip-grad-norm-pre-step`. Running the final beacon cell reports progress against the `Optimizer: clip_grad_norm pre-step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: clip_grad_norm pre-step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`clip-grad-norm-pre-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "clip-grad-norm-pre-step"
DD_SUBTOPIC = "Optimizer: clip_grad_norm pre-step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Optimizer: `clip_grad_norm_` pre-step — quick refresher

Gradient clipping rescales every gradient by the SAME factor when the global L2 norm exceeds `max_norm`. The canonical placement:

```python
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
optimizer.zero_grad()
```

**The math.** Let `g_total = sqrt(sum_i ||g_i||^2)` be the global L2 norm. If `g_total > max_norm`, every param's grad is multiplied by `max_norm / g_total`. Otherwise, no change.

**In-place (note the trailing underscore).** `clip_grad_norm_` modifies `.grad` in place. The return value is the PRE-clip norm — useful for logging.

**MUST be before `optimizer.step()`.** The optimizer reads `.grad`; if you clip after, the step already used the unclipped gradients. Equally bad: clipping after `zero_grad` is a no-op (gradients are gone). Order: backward → clip → step → zero_grad.

**Direction preserved.** Because every grad is scaled by the same scalar, the gradient direction in parameter space is unchanged — only the magnitude. That's the whole point: bound the step size without distorting which way you're going.

**Why not `clip_grad_value_`.** Element-wise clipping (each tensor scaled independently) destroys the direction. Norm-clipping is almost always the right primitive for transformer training.

### Exercise 1 — clip grads to a max global L2 norm before optimizer.step

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `torch.nn.utils.clip_grad_norm_` to rescale a model's gradients to a max global L2 norm BEFORE calling `optimizer.step()`, preserving gradient direction.
> Keywords: gradient-clipping, global-norm, transformer, stability
> ```

**KCs targeted:** `global-l2-norm-rescale`, `clip-pre-step-ordering`

Implement `ex1_clip_and_step(params, optimizer, max_norm)`. The canonical clipped-step utility used in transformer training loops.

1. Compute the GLOBAL L2 norm across every parameter's `.grad` using `torch.nn.utils.clip_grad_norm_`. Pass `max_norm=max_norm`. The function clips in place and returns the PRE-clip norm as a tensor.
2. Call `optimizer.step()` (uses the now-clipped grads).
3. Call `optimizer.zero_grad()` so the next iteration starts fresh.
4. Return the pre-clip norm as a Python float (call `.item()`).

Inputs:
- `params`: iterable of parameter `Tensor`s.
- `optimizer`: a `torch.optim.Optimizer` instance.
- `max_norm`: scalar float.

Output: Python float — the global L2 norm BEFORE clipping.

In [ ]:
import torch.nn.utils as nn_utils

def ex1_clip_and_step(params, optimizer, max_norm: float) -> float:
    """Clip grads to max_norm, step the optimizer, zero grads."""
    raise NotImplementedError()


def _test_ex1():
    # === Hand-computed: when norm > max_norm, all grads scale by max_norm/norm ===
    p1 = t.nn.Parameter(t.zeros(2))
    p2 = t.nn.Parameter(t.zeros(2))
    p1.grad = t.tensor([3.0, 4.0])    # norm = 5
    p2.grad = t.tensor([0.0, 0.0])    # norm = 0
    # Global norm = sqrt(3^2 + 4^2 + 0 + 0) = 5.
    opt = t.optim.SGD([p1, p2], lr=0.0)   # lr=0 so step doesn't move params; we just want clip behavior
    norm = ex1_clip_and_step([p1, p2], opt, max_norm=1.0)
    assert isinstance(norm, float), f'return must be float, got {type(norm).__name__}'
    assert abs(norm - 5.0) < 1e-5, f'expected pre-clip norm=5.0, got {norm}'
    # Optimizer.zero_grad should have cleared the grads (or set them to None).
    for p in (p1, p2):
        if p.grad is not None:
            assert t.allclose(p.grad, t.zeros_like(p.grad)), (
                f'grad should be zeroed after step, got {p.grad}'
            )

    # === Independent setup to verify the clipped step actually applied the clipped grad ===
    p1 = t.nn.Parameter(t.zeros(2))
    p2 = t.nn.Parameter(t.zeros(2))
    p1.grad = t.tensor([3.0, 4.0])
    p2.grad = t.tensor([0.0, 0.0])
    opt = t.optim.SGD([p1, p2], lr=1.0)
    norm = ex1_clip_and_step([p1, p2], opt, max_norm=1.0)
    # After clipping to norm 1: scale = 1/5 = 0.2. So p1.grad became [0.6, 0.8].
    # Step with lr=1.0: p1 = 0 - 1.0 * [0.6, 0.8] = [-0.6, -0.8].
    assert t.allclose(p1.detach(), t.tensor([-0.6, -0.8]), atol=1e-5), (
        f'expected p1=[-0.6, -0.8], got {p1.detach()}'
    )
    assert t.allclose(p2.detach(), t.tensor([0.0, 0.0]), atol=1e-5), (
        f'expected p2 unchanged, got {p2.detach()}'
    )

    # === When norm < max_norm, NO clipping (only step + zero) ===
    p = t.nn.Parameter(t.zeros(2))
    p.grad = t.tensor([0.3, 0.4])  # norm = 0.5
    opt = t.optim.SGD([p], lr=1.0)
    norm = ex1_clip_and_step([p], opt, max_norm=1.0)
    assert abs(norm - 0.5) < 1e-5
    # Step used UNCLIPPED grad: p = 0 - [0.3, 0.4] = [-0.3, -0.4].
    assert t.allclose(p.detach(), t.tensor([-0.3, -0.4]), atol=1e-5)

    # === Direction preserved when clipping (scalar rescale only) ===
    p = t.nn.Parameter(t.zeros(3))
    g_orig = t.tensor([6.0, 8.0, 0.0])  # norm = 10
    p.grad = g_orig.clone()
    opt = t.optim.SGD([p], lr=0.0)
    ex1_clip_and_step([p], opt, max_norm=2.0)
    # Even though zero_grad was called, the clip happened on the cloned grad before step.
    # Verify direction another way: redo without zero, check ratio.
    p = t.nn.Parameter(t.zeros(3))
    p.grad = t.tensor([6.0, 8.0, 0.0])
    nn_utils.clip_grad_norm_([p], max_norm=2.0)
    g_clipped = p.grad
    # Direction == orig direction
    cos = (g_orig @ g_clipped) / (g_orig.norm() * g_clipped.norm())
    assert abs(cos.item() - 1.0) < 1e-5, f'clipping should not rotate gradient, got cos={cos.item()}'
    assert abs(g_clipped.norm().item() - 2.0) < 1e-5, f'clipped norm should be 2.0, got {g_clipped.norm().item()}'

    # === Run a 20-iter loop on a tiny model to check stability under clipping ===
    t.manual_seed(0)
    model = t.nn.Linear(4, 2)
    opt = t.optim.SGD(model.parameters(), lr=10.0)   # crazy LR
    x = t.randn(8, 4)
    y = t.randn(8, 2)
    norms_seen = []
    for _ in range(20):
        loss = (model(x) - y).pow(2).mean()
        loss.backward()
        n = ex1_clip_and_step(list(model.parameters()), opt, max_norm=1.0)
        norms_seen.append(n)
    # Without clipping, lr=10 would diverge. Check the model didn't NaN out.
    for p in model.parameters():
        assert t.isfinite(p).all(), 'model params became non-finite — clipping should have kept us bounded'
    # Pre-clip norms must all be > 0 (something WAS being clipped).
    assert all(n > 0 for n in norms_seen), 'pre-clip norm should always be positive'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
import torch.nn.utils as nn_utils

def ex1_clip_and_step(params, optimizer, max_norm):
    pre_clip_norm = nn_utils.clip_grad_norm_(params, max_norm=max_norm)
    optimizer.step()
    optimizer.zero_grad()
    return pre_clip_norm.item()
```

**`clip_grad_norm_` returns the PRE-clip norm.** A common newbie mistake is to assume the return value is the post-clip norm. It's not — it's what the norm WAS before clipping. The post-clip norm is always `min(pre_clip, max_norm)`.

**Why the function signature takes `params` not the model.** The clip operates on parameter tensors, not on a module hierarchy. `model.parameters()` returns the right iterable. For per-group clipping (different thresholds for the embedding vs the rest), you'd call `clip_grad_norm_` once per group.

**Logging the pre-clip norm is standard practice.** Plotting `grad_norm` over training is one of the most useful debug signals: spikes indicate instability, constant high values mean your `max_norm` is too restrictive, flat-zero means a dead model.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()